# CerberusVision Phase 5.3 — Qwen-2.5-7B-Instruct From-Scratch QLoRA

Bu defter, **sifirdan (from-scratch)** QLoRA fine-tuning yapar.
Phase 5.2'nin truncation (kesilme) hatasini cozer — 16-20 konteynerlik veriler 3072 token sinirini astigi icin
model `<|im_end|>` token'ini goremiyor ve inference'ta sonsuz donguye giriyordu.

**Phase 5.3 Stratejisi:**
- Sifirdan QLoRA — Phase 5.2 adapter YUKLENMEZ
- Konteyner sayisi 14 ile sinirlandirildi (2-14 araligi)
- max_length: 4096 (3072'den yukseltildi — 14 konteyner icin guvenli buffer)
- Coklu Konteyner & Reefer bombardimani: EU/US ondalik format karisik, opsiyonel alanlar
- Tum veri: Phase 5 Base + TR BL + Multi-Con/Reefer + Saf Reefer
- Global aile bazli split — sifir sizinti garantisi
- Erken durdurma: patience=2, eval_steps=10

**Veri (2026-07-25 - Phase 5.3):**
- Multi-Con dagilimi: 30% (2-4), 40% (5-8), 20% (9-11), 10% (12-14)
- Toplam: ~1691 kayit, 212 aile
- Veri dagilimi: %55 Phase 5 / %21 TR BL / %21 Multi-Con / %3 Reefer

**Hedefler:**
- Ekipman: %36 → %60+
- Reefer: %60 → %80+
- TR_Konsimento: %92 korunacak
- Truncation hatasi: %0 (sifir kesilme garantisi)

**Drive Dizini Yapisi:**
```
MyDrive/CerberusVision_Phase5_3_Colab/
├── data/
│   ├── train.jsonl          (~3.5 MB)
│   ├── validation.jsonl     (~700 KB)
│   └── manifest.json
├── checkpoints/          ← egitim sirasinda Drive'a yazilir
└── adapter_best/         ← egitim sonunda buraya kaydedilir
```

In [ ]:
# -U kaldirildi (#181 dersi: agresif upgrade ABI uyumsuzlugu yapabilir)
# TRL araligi sabitlendi (SFTTrainer + processing_class API uyumlulugu)
!pip install -q transformers datasets peft bitsandbytes accelerate "trl>=0.9.6,<1.0"

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
import json

drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/CerberusVision_Phase5_3_Colab")
DATA_DIR = DRIVE_DIR / "data"
CHECKPOINT_DIR = DRIVE_DIR / "checkpoints"
FINAL_ADAPTER_DIR = DRIVE_DIR / "adapter_best"

LOCAL_DATA_DIR = Path("/content/phase5_3_data")

for d in [DATA_DIR, CHECKPOINT_DIR, FINAL_ADAPTER_DIR, LOCAL_DATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Drive mount tamam. Dizinler hazir.")
print(f"  Veri (Drive):        {DATA_DIR}")
print(f"  Checkpoint (Drive):   {CHECKPOINT_DIR}")
print(f"  Final adapter:        {FINAL_ADAPTER_DIR}")
print(f"  Veri (lokal):         {LOCAL_DATA_DIR}")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

model_id = "Qwen/Qwen2.5-7B-Instruct"
use_bf16 = torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

print(f"bf16 supported: {use_bf16}, compute_dtype: {compute_dtype}")

# ============================================================
# Step 1: Load base model (same as Phase 5)
# ============================================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
model = prepare_model_for_kbit_training(model)

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '<|endoftext|>'})

print(f"PAD Token ID: {tokenizer.pad_token_id}")
print(f"EOS Token ID: {tokenizer.eos_token_id}")
assert tokenizer.pad_token_id != tokenizer.eos_token_id, "PAD == EOS!"
assert tokenizer.eos_token == "<|im_end|>", f"EOS token mismatch: {tokenizer.eos_token}"

# ============================================================
# Step 2: Apply fresh LoRA (from-scratch, NOT from Phase 5)
# ============================================================
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("\nFresh LoRA adapter applied — from-scratch training'e hazir.")

In [ ]:
print("Phase 5.3 verileri Drive'dan lokal /content calisma alanina kopyalaniyor...")

expected_files = {
    "train.jsonl": "Phase 5.3 egitim verisi",
    "validation.jsonl": "Phase 5.3 dogrulama verisi",
}

missing = []
for filename, desc in expected_files.items():
    source = DATA_DIR / filename
    destination = LOCAL_DATA_DIR / filename
    if not source.exists():
        missing.append((filename, desc))
        continue
    shutil.copy2(source, destination)
    with open(destination, 'r') as f:
        line_count = sum(1 for _ in f)
    print(f"  Kopyalandi: {filename} ({line_count} satir)")

if missing:
    print("\n*** EKSIK DOSYALAR — Google Drive'a yuklemen gerekiyor ***\n")
    print(f"Hedef klasor: {DATA_DIR}")
    print()
    for filename, desc in missing:
        print(f"  • {filename}  ({desc})")
    print()
    raise FileNotFoundError(f"{len(missing)} dosya Drive'da bulunamadi.")

print("\nVeriler Drive'dan lokal /content calisma alanina tasindi.")

In [ ]:
# ============================================================
# Dataset hazirlama
# ============================================================
train_dataset = load_dataset(
    "json",
    data_files=str(LOCAL_DATA_DIR / "train.jsonl"),
    split="train"
)
eval_dataset = load_dataset(
    "json",
    data_files=str(LOCAL_DATA_DIR / "validation.jsonl"),
    split="train"
)

def format_dataset(example):
    return {
        "prompt": [
            {"role": "system", "content": "Extract shipping instruction data from OCR text as JSON."},
            {"role": "user", "content": str(example['input'])}
        ],
        "completion": [
            {"role": "assistant", "content": str(example['output'])}
        ]
    }

train_dataset = train_dataset.map(format_dataset, remove_columns=train_dataset.column_names)
eval_dataset = eval_dataset.map(format_dataset, remove_columns=eval_dataset.column_names)

print(f"Train size: {len(train_dataset)}")
print(f"Eval size:  {len(eval_dataset)}")
print(f"Train/Eval ratio: {len(eval_dataset)/(len(train_dataset)+len(eval_dataset))*100:.1f}%")

In [ ]:
# ============================================================
# Phase 5.3 Training Config (from-scratch)
# ============================================================
# Kritik degisiklikler vs Phase 5.2:
#   - Konteyner cap: 16-20 → 12-14 (truncation fix)
#   - max_length: 3072 → 4096 (14 konteyner icin guvenli buffer)
#   - eval_steps: 10, patience: 2 (overfitting'i erken yakala)
#   - seed: 3407

RESUME_TRAINING = False  # Runtime koparsa True yap

training_args = SFTConfig(
    output_dir=str(CHECKPOINT_DIR),
    
    # Batch (Phase 5 ile ayni)
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,  # Etkin batch = 16
    
    # LR (Phase 5 ile ayni — from-scratch oldugu icin)
    learning_rate=5e-5,
    num_train_epochs=3,
    warmup_steps=50,
    lr_scheduler_type="cosine",
    
    # Daha sik degerlendirme + erken durdurma
    eval_strategy="steps",
    eval_steps=10,
    save_strategy="steps",
    save_steps=10,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    
    # Logging
    logging_steps=5,
    
    # Donanim
    bf16=use_bf16,
    fp16=not use_bf16,
    optim="paged_adamw_32bit",
    
    # Seed (Phase 5.3)
    seed=3407,
    data_seed=3407,
    
    # SFT ozel
    report_to="none",
    max_length=4096,
    completion_only_loss=True,
    eos_token="<|im_end|>",
    packing=False,
)

print("Phase 5.3 Training Config (from-scratch):")
print(f"  Learning Rate:    {training_args.learning_rate}")
print(f"  Epochs:           {training_args.num_train_epochs}")
print(f"  Batch Size:       {training_args.per_device_train_batch_size} x {training_args.gradient_accumulation_steps} = 16")
print(f"  Eval Steps:       {training_args.eval_steps}")
print("  Early Stopping:   Enabled (via EarlyStoppingCallback, patience=2, threshold=0.001)\n")
print(f"  Max Seq Length:   {training_args.max_length}")
print(f"  Seed:             {training_args.seed}")
print(f"  Resume Training:  {RESUME_TRAINING}")
print()

# ============================================================
# Trainer
# ============================================================
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    args=training_args,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2, early_stopping_threshold=0.001)],
)

# ============================================================
# Train
# ============================================================
print("Egitim basliyor...")
print(f"  Train: {len(train_dataset)} ornek")
print(f"  Eval:  {len(eval_dataset)} ornek")
print(f"  Mode:  FROM-SCRATCH (taze LoRA, Phase 5.2 adapter yok)")
print()

train_result = trainer.train(
    resume_from_checkpoint=True if RESUME_TRAINING else None
)

# ============================================================
# Save adapter
# ============================================================
trainer.model.save_pretrained(str(FINAL_ADAPTER_DIR))
tokenizer.save_pretrained(str(FINAL_ADAPTER_DIR))

# Training metrics
metrics = {
    **train_result.metrics,
    "best_eval_loss": trainer.state.best_metric,
    "best_model_checkpoint": trainer.state.best_model_checkpoint,
    "phase": "5.3",
    "training_type": "from_scratch",
}

metrics_path = FINAL_ADAPTER_DIR / "training_metrics.json"
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print(f"\nPhase 5.3 adapter Drive'a kaydedildi: {FINAL_ADAPTER_DIR}")
print(f"  Best checkpoint: {trainer.state.best_model_checkpoint}")
print(f"  Best eval loss:  {trainer.state.best_metric}")
print(f"  Metrikler:       {metrics_path}")

### Runtime Koparsa Devam Etmek Icin

Eger Colab runtime'i koparsa:

1. **Cell 1'den Cell 6'ya kadar** sirasiyla yeniden calistir
2. **Cell 6'da `RESUME_TRAINING = False` yerine `RESUME_TRAINING = True` yap**
3. Cell 6'yi calistir

```python
RESUME_TRAINING = True   # ← bununla degistir
```

### Egitim Sonrasi

1. `adapter_best/` klasorunu bilgisayara indir
2. Projede `models/Qwen-2.5-7B-Instruct-Phase5_3-LoRA/` olarak kaydet
3. Benchmark'i calistir:
```bash
.venv/bin/python scripts/benchmark_accuracy.py tests/fixtures/qwen_benchmark \
  --output benchmark_results_phase5_3.json \
  --html benchmark_report_phase5_3.html
```